# Sistema Multi-Agent con `smolagents`

In questo notebook costruiamo un piccolo **sistema multi-agent**.
Useremo modelli remoti tramite Hugging Face Inference e tool Python locali per calcoli deterministici. Se il provider gratuito HF e' temporaneamente non disponibile o rate-limited, basta cambiare modello/provider nelle variabili d'ambiente senza riscrivere il notebook.

Architettura della demo:

1. **Manager agent**: riceve il task, pianifica e delega.
2. **Web research agent**: cerca e sintetizza informazioni dal web.
3. **Analysis agent**: usa tool locali per calcolare tempi e punteggi.
4. **Report finale**: il manager combina i risultati in una raccomandazione leggibile.

## 1. Installazione delle Dipendenze

Se stai usando l'ambiente `uv` di questo progetto, le dipendenze principali sono gia' installate in `.venv`.

Se invece apri questo notebook in un ambiente nuovo, decommenta la cella sotto.

In [ ]:
# Decommentare solo se NON si usa l'ambiente uv del progetto
# %pip install "smolagents[gradio,mcp]" duckduckgo-search ddgs python-dotenv huggingface_hub

## 2. Import delle Librerie

Importiamo `smolagents`, `dotenv` e qualche utility standard. I commenti nel codice sono volutamente espliciti: in un sistema multi-agent e' importante capire quali parti sono affidate al modello e quali restano deterministiche.

In [1]:
import math
import os
from typing import Optional, Tuple

from dotenv import load_dotenv
from smolagents import (
    CodeAgent,
    DuckDuckGoSearchTool,
    InferenceClientModel,
    VisitWebpageTool,
    tool,
)

# Carichiamo .env dalla root del progetto.
load_dotenv(".env")

True

## 3. Autenticazione Hugging Face

Il notebook usa `HF_TOKEN` dal file `.env`. Per restare su chiamate remote e non usare GPU locale, useremo `InferenceClientModel`.

Variabili opzionali utili:

- `HF_MODEL_ID`: modello da usare. Default: `Qwen/Qwen2.5-Coder-32B-Instruct`.
- `HF_PROVIDER`: provider HF Inference. Lasciarlo vuoto permette a Hugging Face di scegliere il default disponibile.

Nota pratica: i provider gratuiti possono avere limiti di rate o disponibilita'. Se una cella fallisce per provider/model unavailable, prova un modello piu' piccolo o imposta un provider abilitato sul tuo account HF.

In [2]:
HF_TOKEN = os.getenv("HF_TOKEN")
MODEL_ID = os.getenv("HF_MODEL_ID", "Qwen/Qwen2.5-Coder-32B-Instruct")
HF_PROVIDER = os.getenv("HF_PROVIDER") or None

if HF_TOKEN:
    print(f"HF_TOKEN caricato: {HF_TOKEN[:8]}...")
else:
    print("ATTENZIONE: HF_TOKEN non trovato. Inseriscilo in .env prima di eseguire gli agenti.")

print("Modello:", MODEL_ID)
print("Provider:", HF_PROVIDER or "default Hugging Face")

HF_TOKEN caricato: hf_GxbIC...
Modello: Qwen/Qwen2.5-Coder-32B-Instruct
Provider: default Hugging Face


## 4. Creare il Modello LLM

Tutti gli agenti possono condividere lo stesso modello client. Nulla vieta di usare modelli diversi per ruoli diversi, ma noi restiamo sul semplice.

In [3]:
model = InferenceClientModel(
    model_id=MODEL_ID,
    provider=HF_PROVIDER,
    token=HF_TOKEN,
    timeout=120,
    max_tokens=2048,
)

model

## 5. Tool Locali Deterministici

Gli agenti LLM sono bravi a ragionare e coordinare, ma i calcoli e le regole ripetibili e' meglio affidarli a tool Python, come abbiamo già visto. Qui definiamo due tool:

- `estimate_travel_time`: stima il tempo di viaggio tra coordinate geografiche.
- `score_event_city`: assegna un punteggio semplice a una citta' candidata.

In [ ]:
@tool
def estimate_travel_time(
    origin_coords: Tuple[float, float],
    destination_coords: Tuple[float, float],
    average_speed_kmh: Optional[float] = 180.0,
) -> float:
    """
    Estimate travel time between two points on Earth using great-circle distance.

    Args:
        origin_coords: Tuple with latitude and longitude of the origin.
        destination_coords: Tuple with latitude and longitude of the destination.
        average_speed_kmh: Average speed in km/h. Use lower values for train/car and higher values for flights.

    Returns:
        Estimated travel time in hours, including a 20 percent route overhead.
    """

    lat1, lon1 = map(math.radians, origin_coords)
    lat2, lon2 = map(math.radians, destination_coords)

    earth_radius_km = 6371.0
    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        math.sin(dlat / 2) ** 2
        + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2) ** 2
    )
    distance_km = 2 * earth_radius_km * math.asin(math.sqrt(a))

    # Il moltiplicatore evita una stima troppo ottimistica: le rotte reali non sono linee perfette.
    realistic_distance_km = distance_km * 1.2
    return round(realistic_distance_km / average_speed_kmh, 2)


@tool
def score_event_city(
    travel_hours: float,
    daily_budget_eur: int,
    ai_ecosystem_score: int,
) -> float:
    """
    Score a candidate city for a small AI workshop.

    Args:
        travel_hours: Estimated travel time from the origin city.
        daily_budget_eur: Estimated daily budget per person in euros.
        ai_ecosystem_score: Local AI ecosystem score from 1 to 10.

    Returns:
        A score from 0 to 100 where higher is better.
    """

    travel_score = max(0, 35 - travel_hours * 4) # Penalizza fortemente viaggi lunghi. Ogni ora di viaggio toglie 4 punti ai 35 massimi.
    budget_score = max(0, 35 - max(0, daily_budget_eur - 100) * 0.18) # Budget fino a 100€ al giorno non penalizzato, poi ogni euro in più toglie 0.18 punti ai 35 massimi.
    ecosystem_score = max(1, min(ai_ecosystem_score, 10)) * 3 # Ecosistema AI conta fino a 30 punti, con un minimo di 3 punti anche per i punteggi più bassi.
    return round(travel_score + budget_score + ecosystem_score, 2)


print("Roma -> Milano, treno veloce stimato:", estimate_travel_time((41.9028, 12.4964), (45.4642, 9.19), 180.0), "ore")
print("Score esempio:", score_event_city(travel_hours=3.2, daily_budget_eur=150, ai_ecosystem_score=8))

Roma -> Milano, treno veloce stimato: 3.18 ore
Score esempio: 72.2


## 6. Agente Specializzato: Web Research

Il primo agente specializzato si occupa solo di ricerca web. Gli diamo `DuckDuckGoSearchTool` e `VisitWebpageTool`, ma non i tool di scoring: il suo lavoro e' raccogliere evidenze, non decidere.

In [6]:
web_agent = CodeAgent(
    model=model,
    tools=[DuckDuckGoSearchTool(), VisitWebpageTool()],
    name="web_research_agent",
    description=(
        "Searches the web and summarizes reliable information about cities, "
        "AI communities, venues, and travel context."
    ),
    max_steps=8,
    verbosity_level=1,
)

web_agent

## 7. Agente Specializzato: Analisi

Il secondo agente non naviga il web. Riceve dati e usa tool locali per fare calcoli.

In [7]:
analysis_agent = CodeAgent(
    model=model,
    tools=[estimate_travel_time, score_event_city],
    name="analysis_agent",
    description=(
        "Calculates travel-time estimates and city scores from structured inputs. "
        "It should return compact tables and explain assumptions."
    ),
    additional_authorized_imports=["math", "json"],
    max_steps=8,
    verbosity_level=1,
)

analysis_agent

## 8. Manager Agent

Il manager e' l'orchestratore. Non gli diamo direttamente i tool di ricerca o scoring: li raggiunge delegando agli agenti gestiti. In questo modo il flusso resta leggibile:

- il manager decide la strategia;
- il web agent raccoglie contesto;
- l'analysis agent calcola;
- il manager produce la risposta finale.

In [8]:
manager_agent = CodeAgent(
    model=model,
    tools=[],
    managed_agents=[web_agent, analysis_agent],
    additional_authorized_imports=["json", "pandas"],
    planning_interval=3,
    max_steps=12,
    verbosity_level=2,
)

manager_agent

## 9. Visualizzare la Gerarchia

`visualize()` stampa la struttura del sistema: agenti, tool disponibili e relazioni di delega.

In [9]:
manager_agent.visualize()

CodeAgent | Qwen/Qwen2.5-Coder-32B-Instruct
├── ✅ Authorized imports: ['json', 'pandas']
├── 🛠️ Tools:
│   ┏━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
│   ┃ Name         ┃ Description                                   ┃ Arguments                                    ┃
│   ┡━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│   │ final_answer │ Provides a final answer to the given problem. │ answer (`any`): The final answer to the      │
│   │              │                                               │ problem                                      │
│   └──────────────┴───────────────────────────────────────────────┴──────────────────────────────────────────────┘
└── 🤖 Managed agents:
    ├── web_research_agent | CodeAgent | Qwen/Qwen2.5-Coder-32B-Instruct
    │   ├── ✅ Authorized imports: []
    │   ├── 📝 Description: Searches the web and summarizes reliable information about cities, AI communities, 
    │   │   venues, and travel context.
    │   └── 🛠️ Tools:
    │       ┏━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
    │       ┃ Name          ┃ Description                              ┃ Arguments                                ┃
    │       ┡━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
    │       │ web_search    │ Performs a duckduckgo web search based   │ query (`string`): The search query to    │
    │       │               │ on your query (think a Google search)    │ perform.                                 │
    │       │               │ then returns the top search results.     │                                          │
    │       │ visit_webpage │ Visits a webpage at the given url and    │ url (`string`): The url of the webpage   │
    │       │               │ reads its content as a markdown string.  │ to visit.                                │
    │       │               │ Use this to browse webpages.             │                                          │
    │       │ final_answer  │ Provides a final answer to the given     │ answer (`any`): The final answer to the  │
    │       │               │ problem.                                 │ problem                                  │
    │       └───────────────┴──────────────────────────────────────────┴──────────────────────────────────────────┘
    └── analysis_agent | CodeAgent | Qwen/Qwen2.5-Coder-32B-Instruct
        ├── ✅ Authorized imports: ['math', 'json']
        ├── 📝 Description: Calculates travel-time estimates and city scores from structured inputs. It should 
        │   return compact tables and explain assumptions.
        └── 🛠️ Tools:
            ┏━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
            ┃ Name                 ┃ Description                           ┃ Arguments                            ┃
            ┡━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
            │ estimate_travel_time │ Estimate travel time between two      │ origin_coords (`array`): Tuple with  │
            │                      │ points on Earth using great-circle    │ latitude and longitude of the        │
            │                      │ distance.                             │ origin.                              │
            │                      │                                       │ destination_coords (`array`): Tuple  │
            │                      │                                       │ with latitude and longitude of the   │
            │                      │                                       │ destination.                         │
            │                      │                                       │ average_speed_kmh (`number`):        │
            │                   

## 10. Demo: Scegliere una Citta' per un Workshop AI

 Chiediamo al manager di valutare tre citta' europee per un workshop AI partendo da Roma.

Per tenere il costo basso:

In [11]:
task = """
We are organizing a small two-day AI agents workshop starting from Rome, Italy.

Evaluate these candidate cities: Milan, Paris, and Berlin.

Please delegate:
1. Ask the web_research_agent for concise evidence about each city's AI/startup ecosystem,
   venue/community context, and rough cost level.
2. Ask the analysis_agent to estimate travel time from Rome using these coordinates:
   - Rome: (41.9028, 12.4964)
   - Milan: (45.4642, 9.1900)
   - Paris: (48.8566, 2.3522)
   - Berlin: (52.5200, 13.4050)
   Use 180 km/h for train-like routes and 700 km/h for flight-like routes when appropriate.
3. Ask the analysis_agent to score each city with score_event_city.

Return a compact final report in Italian with:
- a markdown table with city, evidence summary, travel assumption, estimated hours, daily budget, ecosystem score, final score;
- one recommended city;
"""

# Esegui questa cella quando vuoi lanciare davvero il sistema multi-agent.
# Le chiamate passano da Hugging Face e possono richiedere qualche minuto.
result = manager_agent.run(task)
print(result)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ We are organizing a small two-day AI agents workshop starting from Rome, Italy.                                 │
│                                                                                                                 │
│ Evaluate these candidate cities: Milan, Paris, and Berlin.                                                      │
│                                                                                                                 │
│ Please delegate:                                                                                                │
│ 1. Ask the web_research_agent for concise evidence about each city's AI/startup ecosystem,                      │
│    venue/community context, and rough cost level.                                                               │
│ 2. Ask the analysis_agent to estimate travel time from Rome using these coordinates:                            │
│    - Rome: (41.9028, 12.4964)                                                                                   │
│    - Milan: (45.4642, 9.1900)                                                                                   │
│    - Paris: (48.8566, 2.3522)                                                                                   │
│    - Berlin: (52.5200, 13.4050)                                                                                 │
│    Use 180 km/h for train-like routes and 700 km/h for flight-like routes when appropriate.                     │
│ 3. Ask the analysis_agent to score each city with score_event_city.                                             │
│                                                                                                                 │
│ Return a compact final report in Italian with:                                                                  │
│ - a markdown table with city, evidence summary, travel assumption, estimated hours, daily budget, ecosystem     │
│ score, final score;                                                                                             │
│ - one recommended city;                                                                                         │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen2.5-Coder-32B-Instruct ────────────────────────────────────────────────────────╯

────────────────────────────────────────────────── Initial plan ───────────────────────────────────────────────────
Here are the facts I know and the plan of action that I will follow to solve the task:
```
### 1.1. Facts given in the task
- Workshop location: Rome, Italy
- Candidate cities: Milan, Paris, Berlin
- Duration: Two-day workshop
- Requested output language: Italian
- Required elements in final report:
  - Markdown table with city, evidence summary, travel assumption, estimated hours, daily budget, ecosystem score, 
final score
  - One recommended city

### 1.2. Facts to look up
- AI/startup ecosystem strength in Milan, Paris, and Berlin (source: web research)
- Venue/community context in Milan, Paris, and Berlin (source: web research)
- Rough cost level (daily budget) in Milan, Paris, and Berlin (source: web research)
- Coordinates for Rome, Milan, Paris, and Berlin (already provided)
- Travel time estimation methodology (train vs flight, speeds)

### 1.3. Facts to derive
- Estimated travel times between Rome and each candidate city (using provided coordinates and assumed speeds)
- Daily budget estimates for each city (based on cost level research)
- Ecosystem scores for each city (using analysis_agent with score_event_city function)
- Final composite scores for recommendation
- Compact markdown table summarizing all findings
- Recommended city based on scoring

## 2. Plan
1. Delegate to web_research_agent to gather concise evidence about AI/startup ecosystems, venue/community contexts,
and cost levels for Milan, Paris, and Berlin.
2. Delegate to analysis_agent to compute travel times from Rome to each city using provided coordinates and assumed
speeds (180 km/h for trains, 700 km/h for flights).
3. Delegate to analysis_agent to score each city's ecosystem using the score_event_city function.
4. Compile all gathered data into a structured format for analysis.
5. Calculate daily budget estimates for each city based on cost level research.
6. Create a markdown table summarizing city name, evidence summary, travel assumption, estimated hours, daily 
budget, ecosystem score, and final score.
7. Determine the recommended city based on the final scores.
8. Generate final report in Italian including the markdown table and recommendation.
9. Deliver final answer.


```

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
I'll start by delegating the research tasks to gather information about the candidate cities.                      
                                                                                                                   
<code>                                                                                                             
# Task 1: Gather information about AI/startup ecosystems, venue/community context, and costs                       
research_result = web_research_agent(                                                                              
    task="Research the AI/startup ecosystem, venue/community context, and approximate daily cost level for Milan,  
Paris, and Berlin",                                                                                                
    additional_args={}                                                                                             
)                                                                                                                  
print("Research Results:")                                                                                         
print(research_result)                                                                                             
                                                                                                                   

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  # Task 1: Gather information about AI/startup ecosystems, venue/community context, and costs                     
  research_result = web_research_agent(                                                                            
      task="Research the AI/startup ecosystem, venue/community context, and approximate daily cost level for       
  Milan, Paris, and Berlin",                                                                                       
      additional_args={}                                                                                           
  )                                                                                                                
  print("Research Results:")                                                                                       
  print(research_result)                                                                                           
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

...

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
<code>                                                                                                             
final_answer("## Valutazione Città per Workshop AI Agents\n\n## Tabella di Valutazione\n\n| Città | Riepilogo      
Evidenze | Ipotesi di Viaggio | Ore Stimate | Budget Giornaliero | Punteggio Ecosistema | Punteggio Finale         
|\n|-------|-------------------|-------------------|-------------|-------------------|---------------------|-------
-----------|\n| Milano | Ecosistema AI in crescita con forte presenza in fintech, salute e Industry 4.0. Organizza 
eventi come l'AI Festival e Innov8rs Milan. Costi moderati (€80-100/giorno). | Volo | 0.82 h | €80-100 | 45.5 |    
58.5 |\n| Parigi | Centro AI europeo principale con investimenti significativi e aziende come Mistral AI. Comunità 
tecnologica forte con eventi come VivaTech. Costi elevati (€263-290/giorno). | Volo | 1.89 h | €275-313 | 66.5 |   
42.0 |\n| Berlino | Principale hub startup/AI in Germania con cultura innovativa e community vibrante. Costi più   
bassi (€46-100/giorno). | Volo | 2.03 h | €46-100 | 77.5 | 72.0 |\n\n## Raccomandazione\n\n**Berlino è la città    
raccomandata per il workshop.**\n\n### Motivazioni:\n\n1. **Miglior rapporto qualità-prezzo**: Offre un ecosistema 
AI solido con comunità attiva a costi inferiori rispetto a Parigi e Milano.\n\n2. **Accessibilità**: Sebbene il    
viaggio sia più lungo rispetto a Milano, il tempo di volo è ancora ragionevole (2.03 ore).\n\n3. **Costi           
[1;38;2;230;237;243;

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer("## Valutazione Città per Workshop AI Agents\n\n## Tabella di Valutazione\n\n| Città | Riepilogo    
  Evidenze | Ipotesi di Viaggio | Ore Stimate | Budget Giornaliero | Punteggio Ecosistema | Punteggio Finale       
  |\n|-------|-------------------|-------------------|-------------|-------------------|---------------------|---  
  ---------------|\n| Milano | Ecosistema AI in crescita con forte presenza in fintech, salute e Industry 4.0.     
  Organizza eventi come l'AI Festival e Innov8rs Milan. Costi moderati (€80-100/giorno). | Volo | 0.82 h |         
  €80-100 | 45.5 | 58.5 |\n| Parigi | Centro AI europeo principale con investimenti significativi e aziende come   
  Mistral AI. Comunità tecnologica forte con eventi come VivaTech. Costi elevati (€263-290/giorno). | Volo | 1.89  
  h | €275-313 | 66.5 | 42.0 |\n| Berlino | Principale hub startup/AI in Germania con cultura innovativa e         
  community vibrante. Costi più bassi (€46-100/giorno). | Volo | 2.03 h | €46-100 | 77.5 | 72.0 |\n\n##            
  Raccomandazione\n\n**Berlino è la città raccomandata per il workshop.**\n\n### Motivazioni:\n\n1. **Miglior      
  rapporto qualità-prezzo**: Offre un ecosistema AI solido con comunità attiva a costi inferiori rispetto a        
  Parigi e Milano.\n\n2. **Accessibilità**: Sebbene il viaggio sia più lungo rispetto a Milano, il tempo di volo   
  è ancora ragionevole (2.03 ore).\n\n3. **Costi contenuti**: Le spese giornaliere sono significativamente         
  inferiori rispetto alle altre città, consentendo un budget più flessibile per il workshop.\n\n4. **Ecosistema    
  dinamico**: Ha un'elevata valutazione per l'ecosistema AI e la community, rendendola ideale per eventi           
  tecnologici.\n\n5. **Flessibilità**: I costi bassi permettono di allocare risorse aggiuntive al workshop o a     
  esperienze extra per i partecipanti.\n\nSebbene Parigi abbia un ecosistema AI più forte, i costi elevati e il    
  fatto che si tratti di una città molto costosa rendono Berlino la scelta migliore per un workshop di due giorni  
  con budget limitato ma con esigenze di alta qualità.")                                                           
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: ## Valutazione Città per Workshop AI Agents

## Tabella di Valutazione

| Città | Riepilogo Evidenze | Ipotesi di Viaggio | Ore Stimate | Budget Giornaliero | Punteggio Ecosistema | 
Punteggio Finale |
|-------|-------------------|-------------------|-------------|-------------------|---------------------|----------
--------|
| Milano | Ecosistema AI in crescita con forte presenza in fintech, salute e Industry 4.0. Organizza eventi come 
l'AI Festival e Innov8rs Milan. Costi moderati (€80-100/giorno). | Volo | 0.82 h | €80-100 | 45.5 | 58.5 |
| Parigi | Centro AI europeo principale con investimenti significativi e aziende come Mistral AI. Comunità 
tecnologica forte con eventi come VivaTech. Costi elevati (€263-290/giorno). | Volo | 1.89 h | €275-313 | 66.5 | 
42.0 |
| Berlino | Principale hub startup/AI in Germania con cultura innovativa e community vibrante. Costi più bassi 
(€46-100/giorno). | Volo | 2.03 h | €46-100 | 77.5 | 72.0 |

## Raccomandazione

**Berlino è la città raccomandata per il workshop.**

### Motivazioni:

1. **Miglior rapporto qualità-prezzo**: Offre un ecosistema AI solido con comunità attiva a costi inferiori 
rispetto a Parigi e Milano.

2. **Accessibilità**: Sebbene il viaggio sia più lungo rispetto a Milano, il tempo di volo è ancora ragionevole 
(2.03 ore).

3. **Costi contenuti**: Le spese giornaliere sono significativamente inferiori rispetto alle altre città, 
consentendo un budget più flessibile per il workshop.

4. **Ecosistema dinamico**: Ha un'elevata valutazione per l'ecosistema AI e la community, rendendola ideale per 
eventi tecnologici.

5. **Flessibilità**: I costi bassi permettono di allocare risorse aggiuntive al workshop o a esperienze extra per i
partecipanti.

Sebbene Parigi abbia un ecosistema AI più forte, i costi elevati e il fatto che si tratti di una città molto 
costosa rendono Berlino la scelta migliore per un workshop di due giorni con budget limitato ma con esigenze di 
alta qualità.

[Step 6: Duration 7.34 seconds| Input tokens: 36,369 | Output tokens: 1,813]

## Valutazione Città per Workshop AI Agents

## Tabella di Valutazione

| Città | Riepilogo Evidenze | Ipotesi di Viaggio | Ore Stimate | Budget Giornaliero | Punteggio Ecosistema | Punteggio Finale |
|-------|-------------------|-------------------|-------------|-------------------|---------------------|------------------|
| Milano | Ecosistema AI in crescita con forte presenza in fintech, salute e Industry 4.0. Organizza eventi come l'AI Festival e Innov8rs Milan. Costi moderati (€80-100/giorno). | Volo | 0.82 h | €80-100 | 45.5 | 58.5 |
| Parigi | Centro AI europeo principale con investimenti significativi e aziende come Mistral AI. Comunità tecnologica forte con eventi come VivaTech. Costi elevati (€263-290/giorno). | Volo | 1.89 h | €275-313 | 66.5 | 42.0 |
| Berlino | Principale hub startup/AI in Germania con cultura innovativa e community vibrante. Costi più bassi (€46-100/giorno). | Volo | 2.03 h | €46-100 | 77.5 | 72.0 |

## Raccomandazione

**Berlino è la città raccomandata p

## 11. Variante Senza Web Search

Se DuckDuckGo e' rate-limited o vuoi una demo completamente prevedibile, puoi usare solo il manager e l'analysis agent passando dati gia' strutturati. Questo non dimostra la ricerca web, ma dimostra comunque la delega manager -> agente analista.

In [12]:
offline_task = """
We are choosing a city for a small AI agents workshop starting from Rome.

Use only these assumptions, do not browse the web:
- Milan: daily_budget_eur=170, ai_ecosystem_score=8, destination_coords=(45.4642, 9.1900), average_speed_kmh=180
- Paris: daily_budget_eur=220, ai_ecosystem_score=9, destination_coords=(48.8566, 2.3522), average_speed_kmh=700
- Berlin: daily_budget_eur=190, ai_ecosystem_score=8, destination_coords=(52.5200, 13.4050), average_speed_kmh=700
- Origin Rome: (41.9028, 12.4964)

Delegate the calculations to analysis_agent.
Return an Italian markdown table and recommend one city.
"""

# Variante piu' economica e stabile: nessuna ricerca web.
offline_result = manager_agent.run(offline_task)
print(offline_result)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ We are choosing a city for a small AI agents workshop starting from Rome.                                       │
│                                                                                                                 │
│ Use only these assumptions, do not browse the web:                                                              │
│ - Milan: daily_budget_eur=170, ai_ecosystem_score=8, destination_coords=(45.4642, 9.1900),                      │
│ average_speed_kmh=180                                                                                           │
│ - Paris: daily_budget_eur=220, ai_ecosystem_score=9, destination_coords=(48.8566, 2.3522),                      │
│ average_speed_kmh=700                                                                                           │
│ - Berlin: daily_budget_eur=190, ai_ecosystem_score=8, destination_coords=(52.5200, 13.4050),                    │
│ average_speed_kmh=700                                                                                           │
│ - Origin Rome: (41.9028, 12.4964)                                                                               │
│                                                                                                                 │
│ Delegate the calculations to analysis_agent.                                                                    │
│ Return an Italian markdown table and recommend one city.                                                        │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen2.5-Coder-32B-Instruct ────────────────────────────────────────────────────────╯

────────────────────────────────────────────────── Initial plan ───────────────────────────────────────────────────
Here are the facts I know and the plan of action that I will follow to solve the task:
```
### 1.1. Facts given in the task
- Origin city: Rome (coordinates: 41.9028, 12.4964)
- Milan: 
  - daily_budget_eur = 170
  - ai_ecosystem_score = 8
  - destination_coords = (45.4642, 9.1900)
  - average_speed_kmh = 180
- Paris:
  - daily_budget_eur = 220
  - ai_ecosystem_score = 9
  - destination_coords = (48.8566, 2.3522)
  - average_speed_kmh = 700
- Berlin:
  - daily_budget_eur = 190
  - ai_ecosystem_score = 8
  - destination_coords = (52.5200, 13.4050)
  - average_speed_kmh = 700

### 1.2. Facts to look up
None. All required information has been provided directly in the task.

### 1.3. Facts to derive
- Travel time from Rome to each destination city (using average_speed_kmh and coordinates)
- Total cost for a 3-day stay in each city (daily_budget_eur * 3)
- A combined score for each city considering travel time, budget, and AI ecosystem score

## 2. Plan
1. Use analysis_agent to compute the travel time from Rome to Milan, Paris, and Berlin using their coordinates and 
average speeds.
2. Calculate the total cost for a 3-day stay in each city by multiplying the daily budget by 3.
3. Derive a composite score for each city using a weighted combination of travel time, total cost, and AI ecosystem
score.
4. Format the results into an Italian markdown table including city name, travel time, total cost, AI ecosystem 
score, and composite score.
5. Recommend the best city based on the highest composite score.
6. Return the final answer.


```

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
I'll use the analysis_agent to calculate the travel times and then create the recommendation table.                
                                                                                                                   
Let me start by calculating the travel times from Rome to each destination city.                                   
<code>                                                                                                             
# Define the origin and destination coordinates                                                                    
origin = (41.9028, 12.4964)                                                                                        
                                                                                                                   
destinations = {                                                                                                   
    'Milan': (45.4642, 9.1900),                                                                                    
    'Paris': (48.8566, 2.3522),                                                                                    
    'Berlin': (52.5200, 13.4050)                                                                                   
}                                                                                                                  
                                                                                                                   
# Average speeds for each destination                                                                              
speeds = {                                                                                                         
    'Milan': 180,                                                                                                  
    'Paris': 700,                                                                                                  
    'Berlin': 700                                                                                                  
}                                                                                                                  
                                                                                                                   
# Prepare data for analysis agent                                                                                  
analysis_task = """                                                                                                
Calculate the travel time from Rome (41.9028, 12.4964) to each of these destinations:                              
- Milan: (45.4642, 9.1900), average speed 180 km/h                                                                 
- Paris: (48.8566, 2.3522), average speed 700 km/h                                                                 
- Berlin: (52.5200, 13.4050), average speed 700 km/h                                                               
                                                                                                                   
Return the travel times in hours for each destination.                                                             
"""                                                                                                                
                                                                                                                   
# Send to analysis agent                                                                                           
travel_times = analysis_agent(task=analysis_task, additional_args={})                                              
print(travel_times)                                                                                                
                                                        

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  # Define the origin and destination coordinates                                                                  
  origin = (41.9028, 12.4964)                                                                                      
                                                                                                                   
  destinations = {                                                                                                 
      'Milan': (45.4642, 9.1900),                                                                                  
      'Paris': (48.8566, 2.3522),                                                                                  
      'Berlin': (52.5200, 13.4050)                                                                                 
  }                                                                                                                
                                                                                                                   
  # Average speeds for each destination                                                                            
  speeds = {                                                                                                       
      'Milan': 180,                                                                                                
      'Paris': 700,                                                                                                
      'Berlin': 700                                                                                                
  }                                                                                                                
                                                                                                                   
  # Prepare data for analysis agent                                                                                
  analysis_task = """                                                                                              
  Calculate the travel time from Rome (41.9028, 12.4964) to each of these destinations:                            
  - Milan: (45.4642, 9.1900), average speed 180 km/h                                                               
  - Paris: (48.8566, 2.3522), average speed 700 km/h                                                               
  - Berlin: (52.5200, 13.4050), average speed 700 km/h                                                             
                                                                                                                   
  Return the travel times in hours for each destination.                                                           
  """                                                                                                              
                                                                                                                   
  # Send to analysis agent                                                                                         
  travel_times = analysis_agent(task=analysis_task, additional_args={})                                            
  print(travel_times)                                                                                              
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

...

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
<code>                                                                                                             
final_answer("| Città | Tempo di Viaggio | Costo Totale | Punteggio Ecosistema AI | Punteggio Composito            
|\n|---|---|---|---|---|\n| Paris | 1.89 ore | 660 € | 9 | 4.40 |\n| Berlin | 2.03 ore | 570 € | 8 | 4.02 |\n|     
Milan | 3.18 ore | 510 € | 8 | 3.89 |\n\nLa città consigliata è **Paris**, che presenta il punteggio composito più 
alto.")                                                                                                            
                                                                                                                   

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer("| Città | Tempo di Viaggio | Costo Totale | Punteggio Ecosistema AI | Punteggio Composito          
  |\n|---|---|---|---|---|\n| Paris | 1.89 ore | 660 € | 9 | 4.40 |\n| Berlin | 2.03 ore | 570 € | 8 | 4.02 |\n|   
  Milan | 3.18 ore | 510 € | 8 | 3.89 |\n\nLa città consigliata è **Paris**, che presenta il punteggio composito   
  più alto.")                                                                                                      
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: | Città | Tempo di Viaggio | Costo Totale | Punteggio Ecosistema AI | Punteggio Composito |
|---|---|---|---|---|
| Paris | 1.89 ore | 660 € | 9 | 4.40 |
| Berlin | 2.03 ore | 570 € | 8 | 4.02 |
| Milan | 3.18 ore | 510 € | 8 | 3.89 |

La città consigliata è **Paris**, che presenta il punteggio composito più alto.

[Step 3: Duration 2.94 seconds| Input tokens: 12,916 | Output tokens: 1,175]

| Città | Tempo di Viaggio | Costo Totale | Punteggio Ecosistema AI | Punteggio Composito |
|---|---|---|---|---|
| Paris | 1.89 ore | 660 € | 9 | 4.40 |
| Berlin | 2.03 ore | 570 € | 8 | 4.02 |
| Milan | 3.18 ore | 510 € | 8 | 3.89 |

La città consigliata è **Paris**, che presenta il punteggio composito più alto.


## Riepilogo

Abbiamo creato un sistema multi-agent con `smolagents` composto da:

- un **manager agent** che pianifica e delega;
- un **web research agent** con strumenti di ricerca e visita pagine;
- un **analysis agent** con tool Python deterministici;
- una variante offline per testare la delega senza dipendere dal web.

Come si può vedere non sempre tutti gli step vanno a buon fine e non sempre la risposta finale è perfetta. Errori di singoli agenti si propagano al manager agent potenzialmente portando a risultati peggiori. Semplici soluzioni sono: provare con modelli più potenti, modificare il numero di step o planning_interval.